# Report figures
Draws the two figures used in the report from the saved results. Run after Tasks 1–3.

- **Inputs:** `results/task{1,2,3}_metrics.json`, `results/task{2,3}_predictions.json`
- **Outputs:** `results/model_comparison.png`, `results/confusion_matrices.png`
- **Runtime:** CPU, a few seconds.

## Setup

In [ ]:
# Work in a folder on Google Drive so all notebooks share data/ and results/
import os
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/NLP_Group17"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

## Configuration

In [ ]:
import os
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

RESULTS_DIR = "results"
LABELS = ["negative", "neutral", "positive"]

## Helpers

In [ ]:
def load(name):
    with open(os.path.join(RESULTS_DIR, name), "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
def confusion(records):
    m = np.zeros((3, 3), dtype=int)
    for r in records:
        m[LABELS.index(r["gold"]), LABELS.index(r["pred"])] += 1
    return m

## Figures
Test macro-F1 and accuracy for every configuration, and the Task 2 vs. Task 3 confusion matrices.

In [ ]:
def main():
    t1, t2, t3 = load("task1_metrics.json"), load("task2_metrics.json"), load("task3_metrics.json")

    configs = [
        ("Task 1\nRF + norm", t1["task1_test_macro_f1"], t1["task1_test_accuracy"]),
        ("Task 2\nMuRIL raw", t2["task2_test_raw_macro_f1"], t2["task2_test_raw_accuracy"]),
        ("Task 2 check\nnorm at test", t2["task2_test_norm_only_macro_f1"], t2["task2_test_norm_only_accuracy"]),
        ("Task 3\nLID + norm", t3["task3_test_macro_f1"], t3["task3_test_accuracy"]),
    ]
    x = np.arange(len(configs))
    w = 0.38
    fig, ax = plt.subplots(figsize=(8, 4.2))
    b1 = ax.bar(x - w / 2, [c[1] for c in configs], w, label="Test macro-F1", color="#3b6ea5")
    b2 = ax.bar(x + w / 2, [c[2] for c in configs], w, label="Test accuracy", color="#9bb8d3")
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.005, f"{b.get_height():.3f}",
                    ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([c[0] for c in configs], fontsize=9)
    ax.set_ylim(0.5, 0.65)
    ax.set_ylabel("Score")
    ax.set_title("SentiMix Partition B test split (n = 616)")
    ax.legend(loc="upper left", fontsize=9, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, "model_comparison.png"), dpi=200)
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for ax, (title, name) in zip(axes, [("Task 2: MuRIL (raw)", "task2_predictions.json"),
                                       ("Task 3: LID-augmented MuRIL", "task3_predictions.json")]):
        m = confusion(load(name))
        ax.imshow(m, cmap="Blues", vmin=0, vmax=m.max())
        for i in range(3):
            for j in range(3):
                ax.text(j, i, m[i, j], ha="center", va="center",
                        color="white" if m[i, j] > m.max() / 2 else "black", fontsize=11)
        ax.set_xticks(range(3))
        ax.set_xticklabels(LABELS)
        ax.set_yticks(range(3))
        ax.set_yticklabels(LABELS)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Gold")
        ax.set_title(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, "confusion_matrices.png"), dpi=200)
    plt.close(fig)
    print("Saved model_comparison.png and confusion_matrices.png to", RESULTS_DIR)

## Run

In [ ]:
main()

from IPython.display import Image, display
for name in ["model_comparison.png", "confusion_matrices.png"]:
    display(Image(os.path.join(RESULTS_DIR, name)))